# DQ Summaries & Review Package (PIP — 02-Nov task)

**Deliverable:** plain-language DQ summaries generated from **actual rule-based DQ results**, with every number traceable to a query in this notebook.

## Method labeling (required by the plan)

| Component | Method |
| --- | --- |
| DQ gate DQ-01…DQ-07 | **Rule-based** (deterministic PySpark rules in `src/dq_checks.py`) |
| Quarantine routing (Silver) | **Rule-based** (dedup on `subject_id`, null-key rejection, referential `left_anti`) |
| Plain-language summaries below | **Rule-based** — written from executed metrics; **no LLM was used** to produce the numbers |
| LLM-assisted phrasing (optional stretch) | **Not used** — free AI API not exercised (plan allows this) |

**How to reproduce:** run the cells top-to-bottom on any compute with access to `workspace.bronze.*` / `workspace.silver.*`. The gate re-executes live; numbers below were captured 02-Nov-2026.

## Result snapshot (executed live in Cell 2)

- DQ gate: **7/7 PASS**, 0 violations
- Bronze → Silver reconciliation: subjects 280 → 267 valid + 5 quarantined + 8 duplicate rows dropped; visits 1492 → 1463 valid + 29 orphan visits quarantined

In [0]:
# Cell 2 — run the REAL rule-based DQ gate (src/dq_checks.py) on live Silver tables
import sys
sys.path.insert(0, "/Workspace/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip")
from src.dq_checks import run_dq_checks, summarize

subjects = spark.table("workspace.silver.subjects")
visits   = spark.table("workspace.silver.visits")
studies  = spark.table("workspace.silver.studies")
sites    = spark.table("workspace.silver.sites")

checks = run_dq_checks(subjects, visits, studies, sites)
print(summarize(checks))

import pandas as pd
display(pd.DataFrame(checks))

In [0]:
# Cell 3 — traceability metrics: every number the summaries cite
from pyspark.sql import functions as F

bronze_counts = {t: spark.table(f"workspace.bronze.{t}").count()
                 for t in ["studies", "sites", "subjects", "visits"]}
silver_counts = {t: spark.table(f"workspace.silver.{t}").count()
                 for t in ["studies", "sites", "subjects", "visits"]}
q_subj = spark.table("workspace.silver.subjects_quarantine").count()
q_vis  = spark.table("workspace.silver.visits_quarantine").count()

b_subj = spark.table("workspace.bronze.subjects")
dup_keys = b_subj.groupBy("subject_id").count().filter("count > 1").count()
dup_rows = b_subj.count() - b_subj.select("subject_id").distinct().count()
null_site = b_subj.filter(F.col("site_id").isNull()).count()

print(f"Bronze rows  : {bronze_counts}")
print(f"Silver rows  : {silver_counts}")
print(f"Quarantine   : subjects={q_subj}, visits={q_vis}")
print(f"Bronze subject issues: duplicate subject_id keys={dup_keys} (extra rows={dup_rows}), null site_id rows={null_site}")
print("Reconciliation subjects: bronze", bronze_counts["subjects"],
      "=", silver_counts["subjects"], "valid +", q_subj, "quarantined +", dup_rows, "dups dropped",
      "->", silver_counts["subjects"] + q_subj + dup_rows)
print("Reconciliation visits  : bronze", bronze_counts["visits"],
      "=", silver_counts["visits"], "valid +", q_vis, "quarantined")

In [0]:
# Cell 4 — Case 1 evidence: duplicate subject_ids in Bronze, removed by Silver dedup
from pyspark.sql import functions as F
b = spark.table("workspace.bronze.subjects")
dups = (b.groupBy("subject_id").count().filter("count > 1")
         .orderBy(F.desc("count")))
print("Duplicated subject_id keys in bronze.subjects:", dups.count())
display(dups)

In [0]:
# Cell 5 — Case 2 evidence: rows with NULL site_id, quarantined before reaching Silver
from pyspark.sql import functions as F
b = spark.table("workspace.bronze.subjects")
nulls = b.filter(F.col("site_id").isNull()).select("subject_id", "study_id", "site_id", "subject_status")
print("Bronze subjects with NULL site_id:", nulls.count())
display(nulls)
q = spark.table("workspace.silver.subjects_quarantine").select("subject_id", "site_id", "silver_load_timestamp")
print("Rows in silver.subjects_quarantine:", q.count())
display(q)
# Silver must be clean
print("Silver subjects with blank/null site_id:",
      spark.table("workspace.silver.subjects").filter(F.col("site_id").isNull() | (F.trim("site_id") == "")).count())

In [0]:
# Cell 6 — Case 3 evidence: orphan visits quarantined with explicit reason
from pyspark.sql import functions as F
vq = spark.table("workspace.silver.visits_quarantine")
print("Orphan visits quarantined:", vq.count())
display(vq.groupBy("quarantine_reason").count().orderBy(F.desc("count")))
display(vq.select("visit_id", "subject_id", "visit_name", "quarantine_reason").limit(10))
# cascade check: do the orphan visits belong to the quarantined subjects?
bad_subjects = [r.subject_id for r in spark.table("workspace.silver.subjects_quarantine").select("subject_id").collect()]
cascade = vq.filter(F.col("subject_id").isin(bad_subjects)).count()
print(f"Orphan visits whose subject is a quarantined subject: {cascade} of {vq.count()}")

In [0]:
# Cell 7 — reconciliation proof: bronze = silver valid + quarantine + duplicates dropped
from pyspark.sql import functions as F
b = spark.table("workspace.bronze.subjects")
s = spark.table("workspace.silver.subjects")
q = spark.table("workspace.silver.subjects_quarantine")
bv, sv, qv = b.count(), s.count(), q.count()
dups_dropped = bv - b.select("subject_id").distinct().count()
print(f"subjects: bronze {bv} = silver {sv} + quarantined {qv} + dups {dups_dropped} -> {sv + qv + dups_dropped == bv}")
bv2 = spark.table("workspace.bronze.visits").count()
qv2 = spark.table("workspace.silver.visits_quarantine").count()
print(f"visits  : bronze {bv2} = silver {sv and spark.table('workspace.silver.visits').count()} + quarantined {qv2} -> {spark.table('workspace.silver.visits').count() + qv2 == bv2}")

## Plain-language DQ case summaries

*(All numbers produced by the rule-based checks and queries above — executed 02-Nov-2026. Method: rule-based only; no LLM used.)*

### Case 1 — Duplicate patient records would have double-counted enrollments
> "The raw feed contained 280 subject rows, but 8 `subject_id` values appeared more than once (8 extra rows). If loaded as-is, those patients would be counted twice in every KPI. The Silver dedup rule keeps one row per `subject_id`, so the clean table holds **267 unique subjects**, and gate check **DQ-01 confirms 0 duplicates**. Nothing was lost — the reconciliation proves 267 + 5 quarantined + 8 duplicates = 280 original rows."

- **Check:** DQ-01 (duplicate `subject_id` in `silver.subjects`) — **PASS, violations=0**
- **Source metrics:** Cell 3 (bronze 280, dup keys 8) · Cell 4 (duplicate keys listed) · Cell 7 (reconciliation)

### Case 2 — Missing site IDs were caught before they could corrupt site KPIs
> "5 subjects arrived with **no site_id at all** (e.g. `SUB-0034`, `SUB-0109`). A subject without a site cannot be attributed to any site's enrollment KPI, so these rows are **quarantined, not silently dropped** — they sit in `silver.subjects_quarantine` with their data intact for inspection. The published `silver.subjects` table has **0 rows with blank/null site_id** (check **DQ-02: PASS**), so site-level KPIs are computed only on attributable subjects."

- **Check:** DQ-02 (non-blank `site_id`) — **PASS, violations=0**
- **Source metrics:** Cell 3 (null site_id rows = 5) · Cell 5 (quarantine rows + silver clean proof)

### Case 3 — Orphan visits were quarantined as a cascade of Case 2
> "29 visits referenced subjects that are not in the clean Silver table. These are not random defects: **all 29 belong to the 5 subjects quarantined in Case 2** — when a subject is rejected, its visits are quarantined with it, and each row carries an explicit reason (`subject_id not in silver.subjects`). Result: `silver.visits` contains only **1,463 visits that join cleanly to a real subject** (check **DQ-07: PASS**), so visit KPIs never count phantom patients."

- **Check:** DQ-07 (every visit has a subject) — **PASS, violations=0**
- **Source metrics:** Cell 3 (visits 1492 → 1463 + 29) · Cell 6 (reason breakdown + cascade proof)

### Other gate results (same method)
DQ-03, DQ-04, DQ-05, DQ-06 — **PASS, 0 violations** (mandatory keys present; subject→study referential integrity intact). Full list in Cell 2.

In [0]:
# Cell 9 — unit-test proof that the DQ rules behave correctly (rule-based, deterministic)
import sys, os, shutil, tempfile, pytest
src = "/Workspace/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip"
dst = tempfile.mkdtemp(prefix="ctdq-")
repo = os.path.join(dst, "repo")
shutil.copytree(src, repo, ignore=shutil.ignore_patterns("__pycache__", ".ipynb_checkpoints", "*.ipynb"))
sys.path.insert(0, repo)
os.chdir(repo)
# Re-run safety: purge test modules cached by a previous run of this cell in the
# same kernel session — otherwise pytest sees an "import file mismatch" between
# the old and new temp copies and aborts collection (exit code 2).
for _m in [m for m in list(sys.modules) if m.startswith(("test_", "tests."))]:
    del sys.modules[_m]
rc = pytest.main(["-q", "-p", "no:cacheprovider", "--import-mode=importlib", "tests"])
print("pytest exit code:", rc, "(0 = all passed; 14 tests incl. negative test 'missing required column')")